# D33 | PM | Take-Home | SVM, KNN & Full Week Comparison
**Day 33 | Week 6 | Machine Learning & AI**

---

## Part A: 8-Algorithm Comparison Notebook

In [ ]:
# Imports needed for the entire notebook
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# All 8 algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

print("All imports successful.")

### Algorithm Cards

In [ ]:
# I'm printing the algorithm cards as a reference before running the models

algo_cards = {
    "Logistic Regression": {
        "when": "Binary/multiclass classification, interpretability needed, linearly separable data",
        "params": "C (regularization), max_iter, solver, penalty",
        "pros": "Fast, interpretable, works well on small datasets, outputs probabilities",
        "cons": "Assumes linearity, poor with complex relationships",
        "code": "LogisticRegression(C=1.0, max_iter=1000).fit(X_train, y_train)"
    },
    "Decision Tree": {
        "when": "Need interpretability, mixed data types, non-linear boundaries",
        "params": "max_depth, min_samples_split, criterion, max_features",
        "pros": "Highly interpretable, no scaling needed, handles mixed types",
        "cons": "Overfits easily, unstable (small data change = big tree change)",
        "code": "DecisionTreeClassifier(max_depth=5, criterion='gini').fit(X_train, y_train)"
    },
    "Random Forest": {
        "when": "High-dimensional tabular data, robust baseline needed, feature importance",
        "params": "n_estimators, max_depth, max_features, min_samples_leaf",
        "pros": "Reduces overfitting vs DT, handles missing values, feature importance",
        "cons": "Less interpretable, slower than single DT, large memory for big forests",
        "code": "RandomForestClassifier(n_estimators=100, max_depth=10).fit(X_train, y_train)"
    },
    "Gradient Boosting": {
        "when": "Tabular data competitions, structured data, need high accuracy",
        "params": "n_estimators, learning_rate, max_depth, subsample",
        "pros": "Usually best accuracy on tabular data, handles mixed types",
        "cons": "Slow to train, many hyperparameters, can overfit if not tuned",
        "code": "GradientBoostingClassifier(n_estimators=100, learning_rate=0.1).fit(X_train, y_train)"
    },
    "SVM": {
        "when": "High-dim data, text classification, clear margin of separation",
        "params": "C, kernel (rbf/linear/poly), gamma",
        "pros": "Works well in high dimensions, robust to outliers with right C",
        "cons": "Slow on large datasets, sensitive to feature scaling, kernel choice matters",
        "code": "SVC(C=1.0, kernel='rbf', gamma='scale').fit(X_train, y_train)"
    },
    "KNN": {
        "when": "Small datasets, non-linear boundaries, need simple baseline",
        "params": "n_neighbors (K), metric, weights",
        "pros": "No training time, intuitive, naturally handles multi-class",
        "cons": "Slow prediction on large data, sensitive to irrelevant features and scale",
        "code": "KNeighborsClassifier(n_neighbors=5, metric='minkowski').fit(X_train, y_train)"
    },
    "Naive Bayes": {
        "when": "Text classification, very small datasets, need fast baseline",
        "params": "var_smoothing (GaussianNB), alpha (MultinomialNB)",
        "pros": "Extremely fast, works with small data, strong baseline for text",
        "cons": "Strong independence assumption rarely holds, poor probability calibration",
        "code": "GaussianNB(var_smoothing=1e-9).fit(X_train, y_train)"
    },
    "XGBoost": {
        "when": "Tabular data competitions, need best accuracy, handles missing values natively",
        "params": "n_estimators, learning_rate, max_depth, reg_alpha, reg_lambda",
        "pros": "Fast, handles missing values, built-in regularization, very accurate",
        "cons": "Many hyperparameters, less interpretable than LR/DT, can overfit",
        "code": "XGBClassifier(n_estimators=200, learning_rate=0.1, use_label_encoder=False).fit(X_train, y_train)"
    }
}

for name, card in algo_cards.items():
    print(f"=== {name} ===")
    for key, val in card.items():
        print(f"  {key.capitalize()}: {val}")
    print()

### Load Dataset & Run All 8 Algorithms with CV

In [ ]:
# Using breast cancer dataset - binary classification, 30 features, 569 samples
data = load_breast_cancer()
X, y = data.data, data.target

print(f"Dataset: Breast Cancer")
print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"Classes: {data.target_names}")

In [ ]:
# Define all 8 models - wrapping in pipelines so scaling is consistent
models = {
    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000))
    ]),
    "Decision Tree": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', DecisionTreeClassifier(max_depth=5, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    "Gradient Boosting": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=100, random_state=42))
    ]),
    "SVM (RBF)": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(C=1.0, kernel='rbf', gamma='scale'))
    ]),
    "KNN (K=5)": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ]),
    "Naive Bayes": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GaussianNB())
    ]),
    "XGBoost": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42,
                               eval_metric='logloss', verbosity=0))
    ])
}

# 5-fold stratified CV for fair comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for model_name, pipeline in models.items():
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
    results[model_name] = {
        'mean_accuracy': scores.mean(),
        'std': scores.std(),
        'scores': scores
    }
    print(f"{model_name:25s} | CV Acc: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# Rank all models
results_df = pd.DataFrame([
    {'Algorithm': name, 'Mean CV Accuracy': v['mean_accuracy'], 'Std Dev': v['std']}
    for name, v in results.items()
]).sort_values('Mean CV Accuracy', ascending=False).reset_index(drop=True)

results_df.index += 1  # rank starts at 1
results_df.index.name = 'Rank'
print(results_df.to_string())

best_model = results_df.iloc[0]
print(f"\nBest Model: {best_model['Algorithm']} with CV Accuracy = {best_model['Mean CV Accuracy']:.4f}")
print("""
Reasoning: SVM with RBF kernel (or XGBoost) typically wins on breast cancer data.
- The dataset is medium-sized (569 samples, 30 features) — ideal for SVM.
- Features are continuous and benefit from scaling, which we did in the pipeline.
- SVM with RBF maps to higher dimension and finds a good margin.
- XGBoost is close because boosting handles subtle feature interactions well.
""")

---
## Part B: SVM for Text Classification (TF-IDF + LinearSVC Pipeline)

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression as LR
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# 4 news categories chosen
categories = [
    'sci.med',
    'sci.space',
    'talk.politics.guns',
    'rec.sport.hockey'
]

# Load train and test splits
train_data = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'))
test_data  = fetch_20newsgroups(subset='test',  categories=categories,
                                 remove=('headers', 'footers', 'quotes'))

X_train_text, y_train_text = train_data.data, train_data.target
X_test_text,  y_test_text  = test_data.data,  test_data.target

print(f"Train samples: {len(X_train_text)}, Test samples: {len(X_test_text)}")
print(f"Categories: {train_data.target_names}")

In [ ]:
# TF-IDF + LinearSVC pipeline
svm_text_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, sublinear_tf=True, ngram_range=(1,2))),
    ('clf', LinearSVC(C=1.0, max_iter=2000))
])

svm_text_pipeline.fit(X_train_text, y_train_text)
svm_preds = svm_text_pipeline.predict(X_test_text)

print("=== TF-IDF + LinearSVC ===")
print(f"Test Accuracy: {accuracy_score(y_test_text, svm_preds):.4f}")
print(classification_report(y_test_text, svm_preds, target_names=train_data.target_names))

In [ ]:
# Compare with Logistic Regression on the same text data
lr_text_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, sublinear_tf=True, ngram_range=(1,2))),
    ('clf', LR(C=1.0, max_iter=2000, solver='saga'))
])

lr_text_pipeline.fit(X_train_text, y_train_text)
lr_preds = lr_text_pipeline.predict(X_test_text)

print("=== TF-IDF + Logistic Regression ===")
print(f"Test Accuracy: {accuracy_score(y_test_text, lr_preds):.4f}")
print(classification_report(y_test_text, lr_preds, target_names=train_data.target_names))

print("\n--- Summary ---")
print(f"LinearSVC Accuracy : {accuracy_score(y_test_text, svm_preds):.4f}")
print(f"Logistic Regression: {accuracy_score(y_test_text, lr_preds):.4f}")
print("""
Observation:
- LinearSVC is usually marginally better or equal to LR on text data.
- Both are fast on TF-IDF vectors since the data is linearly separable in high-dim TF-IDF space.
- LR gives probability outputs while LinearSVC doesn't (needs Platt scaling for probabilities).
""")

---
## Part C: Interview Ready

### Q1: 100 Features, 50 Samples — Which algorithms work?

**Answer:**

With 100 features and only 50 samples, we are in a **high-dimensional, low-sample** situation (p >> n).

**Would work well:**
- **Logistic Regression with L1/L2 regularization** — regularization prevents overfitting; L1 also does feature selection automatically.
- **SVM with linear kernel** — works well in high dimensions because it only relies on support vectors (a small subset), not all samples. The margin maximization acts as a regularizer.
- **Naive Bayes** — despite its independence assumption, it works surprisingly well with few samples because it has very few parameters to estimate.

**Would likely fail or be unreliable:**
- **Decision Tree / Random Forest** — with 50 samples, trees overfit badly. RF helps somewhat but 50 samples is still too few for reliable splits.
- **KNN** — suffers severely from the curse of dimensionality. In 100 dimensions, all points are roughly equidistant, so the notion of "nearest" breaks down.
- **XGBoost / Gradient Boosting** — boosting builds many trees, each overfitting residuals. With 50 samples this almost certainly leads to overfitting unless very aggressively regularized.
- **SVM with RBF kernel** — flexible kernel can memorize 50 samples perfectly; needs heavy regularization (high C penalty with tuned gamma).

**Recommendation:** Use Logistic Regression with L1 (`penalty='l1'`) — it selects the most relevant features automatically and generalizes well.

### Q2: model_selection_report() function

In [ ]:
from scipy.stats import ttest_rel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
import pandas as pd
import numpy as np

def model_selection_report(X, y, models_dict, n_splits=5, random_state=42):
    """
    Runs stratified CV for each model, returns a DataFrame with metrics,
    and identifies the statistically best model using paired t-test.

    Parameters:
        X          : feature matrix
        y          : target labels
        models_dict: dict of {model_name: sklearn estimator/pipeline}
        n_splits   : number of CV folds
        random_state: for reproducibility

    Returns:
        summary_df : DataFrame with CV scores per model
    """
    # Scale data — models that don't need it won't be hurt
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    cv_strategy = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    all_scores = {}

    for name, model in models_dict.items():
        fold_scores = cross_val_score(model, X_scaled, y, cv=cv_strategy, scoring='accuracy')
        all_scores[name] = fold_scores

    # Build summary table
    rows = []
    for name, scores in all_scores.items():
        rows.append({
            'Model': name,
            'Mean Accuracy': round(scores.mean(), 4),
            'Std Dev': round(scores.std(), 4),
            'Min': round(scores.min(), 4),
            'Max': round(scores.max(), 4)
        })

    summary_df = pd.DataFrame(rows).sort_values('Mean Accuracy', ascending=False).reset_index(drop=True)

    # Identify best model
    best_name = summary_df.iloc[0]['Model']
    best_scores = all_scores[best_name]

    print("=== Model Selection Report ===")
    print(summary_df.to_string(index=False))
    print(f"\nBest model by mean CV accuracy: {best_name}")

    # Paired t-test: best vs all others
    print("\n--- Paired t-test: Best vs Others ---")
    for name, scores in all_scores.items():
        if name == best_name:
            continue
        t_stat, p_val = ttest_rel(best_scores, scores)
        significance = "*significant*" if p_val < 0.05 else "not significant"
        print(f"  {best_name} vs {name}: p={p_val:.4f} ({significance})")

    return summary_df


# Quick test using breast cancer data
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

X_bc, y_bc = load_breast_cancer(return_X_y=True)

test_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', gamma='scale')
}

report_df = model_selection_report(X_bc, y_bc, test_models)

### Q3: SVM(RBF) — train accuracy 1.0, test accuracy 0.52

**Problem:** Classic overfitting. The model has memorized the training data but learned nothing generalizable.

**Why it happens with SVM(RBF):** The RBF kernel can create very complex decision boundaries. When `C` is too large or `gamma` is too high, the model can perfectly separate every training point — fitting noise instead of the signal.

**3 Specific Fixes:**

1. **Reduce `C` (regularization strength):** Lower `C` forces a softer margin — the model tolerates more training misclassifications but generalizes better. Try `C` in [0.001, 0.01, 0.1] using GridSearchCV.

2. **Reduce `gamma`:** High `gamma` means the kernel function has very small radius of influence — each training point essentially becomes its own island. Use `gamma='scale'` or `gamma='auto'` instead of a manually large value. This smooths the decision boundary.

3. **Use proper cross-validation for hyperparameter tuning:** The model was likely selected or tuned on the test set (data leakage) or without CV. Use `GridSearchCV` or `RandomizedSearchCV` with 5-fold CV to pick `C` and `gamma`, and only evaluate once on a truly held-out test set.

**Bonus fix:** Ensure feature scaling (StandardScaler) is applied — unscaled features can cause gamma to behave unexpectedly.

---
## Part D: AI-Augmented — Algorithm Selection Decision Guide

*(I used AI to generate an initial decision framework, then verified and extended it based on week's learning.)*

```
ALGORITHM SELECTION GUIDE — Week 6 ML Algorithms
=================================================

START
 |
 +-- Is it TEXT data?
 |    YES --> TF-IDF + LinearSVC (or Logistic Regression if probabilities needed)
 |    NO  --> Continue
 |
 +-- Dataset size?
 |    SMALL (<500 samples)
 |     |
 |     +-- High dimensions (features > samples)?
 |     |    YES --> Logistic Regression (L1/L2) or SVM Linear
 |     |    NO  --> Naive Bayes (baseline), KNN, or Logistic Regression
 |     |
 |    MEDIUM (500–50K samples)
 |     |
 |     +-- Need interpretability?
 |     |    YES --> Logistic Regression or Decision Tree
 |     |    NO  --> SVM (RBF), Random Forest, or XGBoost
 |     |
 |    LARGE (>50K samples)
 |     |
 |     --> XGBoost or Random Forest (SVM too slow)
 |
 +-- Is data linearly separable?
 |    YES --> Logistic Regression, SVM Linear
 |    NO  --> SVM RBF, Random Forest, Gradient Boosting, XGBoost
 |
 +-- Need feature importance?
      YES --> Random Forest or XGBoost (both give .feature_importances_)
      NO  --> Any of the above

EDGE CASES THE AI MISSED (verified from experience):
- Very imbalanced data: All models need class_weight='balanced' or SMOTE preprocessing
- Time series: None of these are directly applicable without feature engineering
- Noisy labels: SVM is sensitive; Random Forest / XGBoost are more robust
- Deployment latency: KNN is slow at predict time; LR and NB are fastest
- Probability calibration needed: SVM probabilities are poorly calibrated by default
```